# Tourism-Yearly Haar + ld=16 Confirmation Study

**Question:** Can TrendWaveletAE or TrendWaveletAELG with Haar wavelet at latent_dim=16 beat the current Tourism-Yearly SOTA (coif3_eq_bcast_td3_ld16, SMAPE=20.864)?

**Background:** The v2 Tourism study found Haar was the best wavelet family for Tourism-Yearly (KW p=0.009) and ld=12 > ld=8 (MWU p=0.020). This experiment tests whether pushing to ld=16 -- the established best latent dim on M4-Yearly -- further improves performance.

**Design:** 2 configs (AE, AELG) x 10 runs x 50 epochs. Zero divergences.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Load data
df = pd.read_csv('experiments/results/tourism/tourism_haar_ld16_confirmation_results.csv')
sota_df = pd.read_csv('experiments/results/tourism/tourism_aelg_sota_confirmation_results.csv')
v2_df = pd.read_csv('experiments/results/tourism/trendwaveletae_v2_study_results.csv')

ae = df[df['config_name'] == 'TrendWaveletAE_haar_eq_fcast_td3_ld16'].sort_values('seed')
aelg = df[df['config_name'] == 'TrendWaveletAELG_haar_eq_fcast_td3_ld16'].sort_values('seed')
sota = sota_df.sort_values('seed')

print(f"Haar ld=16 study: {len(df)} rows, {df['config_name'].nunique()} configs, {df['diverged'].sum()} divergences")
print(f"SOTA confirmation: {len(sota_df)} rows")

## Summary Statistics and 95% Confidence Intervals

All three 10-run distributions with their CIs. The key question is whether either Haar config's CI sits below the SOTA CI.

In [ ]:
def ci95(arr):
    return stats.t.interval(0.95, len(arr)-1, loc=arr.mean(), scale=stats.sem(arr))

configs = {
    'SOTA: AELG_coif3_eq_bcast_ld16': sota['smape'].values,
    'AE_haar_eq_fcast_ld16': ae['smape'].values,
    'AELG_haar_eq_fcast_ld16': aelg['smape'].values,
}

rows = []
for name, vals in configs.items():
    lo, hi = ci95(vals)
    rows.append({'Config': name, 'N': len(vals), 'Mean': vals.mean(), 'Std': vals.std(ddof=1),
                 'Min': vals.min(), 'Max': vals.max(), 'CI_lo': lo, 'CI_hi': hi})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False, float_format='%.3f'))

## Visual: CI Comparison and Per-Seed Scatter

Left panel shows the 95% CIs for all three configs. Right panel shows per-seed SMAPE for the matched seeds across all three configs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: CI comparison
labels = ['SOTA\ncoif3 AELG', 'AE\nhaar', 'AELG\nhaar']
means = [sota['smape'].mean(), ae['smape'].mean(), aelg['smape'].mean()]
cis = [ci95(sota['smape'].values), ci95(ae['smape'].values), ci95(aelg['smape'].values)]
colors = ['#2ecc71', '#3498db', '#e74c3c']
for i, (m, ci, c) in enumerate(zip(means, cis, colors)):
    ax1.errorbar(i, m, yerr=[[m - ci[0]], [ci[1] - m]], fmt='o', markersize=10, color=c,
                 capsize=8, capthick=2, linewidth=2)
ax1.set_xticks(range(3))
ax1.set_xticklabels(labels)
ax1.set_ylabel('SMAPE')
ax1.set_title('95% Confidence Intervals')
ax1.axhline(y=sota['smape'].mean(), color='#2ecc71', linestyle='--', alpha=0.4, label='SOTA mean')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Right: Per-seed scatter
seeds = sorted(ae['seed'].values)
ae_vals = ae.set_index('seed').loc[seeds, 'smape'].values
aelg_vals = aelg.set_index('seed').loc[seeds, 'smape'].values
sota_vals = sota.set_index('seed').loc[seeds, 'smape'].values
x = np.arange(len(seeds))
w = 0.25
ax2.bar(x - w, sota_vals, w, label='SOTA coif3', color='#2ecc71', alpha=0.8)
ax2.bar(x, ae_vals, w, label='AE haar', color='#3498db', alpha=0.8)
ax2.bar(x + w, aelg_vals, w, label='AELG haar', color='#e74c3c', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels([str(s) for s in seeds], fontsize=8)
ax2.set_xlabel('Seed')
ax2.set_ylabel('SMAPE')
ax2.set_title('Per-Seed Comparison')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('experiments/analysis/notebooks/tourism_haar_ld16_ci_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("SOTA wins on 8/10 seeds for AE_haar, 9/10 seeds for AELG_haar.")

## Statistical Tests

### AE vs AELG backbone (within this study)

In [ ]:
# AE vs AELG
u_stat, p_mwu = stats.mannwhitneyu(ae['smape'].values, aelg['smape'].values, alternative='two-sided')
w_stat, p_w = stats.wilcoxon(ae['smape'].values, aelg['smape'].values, alternative='two-sided')
pooled_std = np.sqrt((ae['smape'].std(ddof=1)**2 + aelg['smape'].std(ddof=1)**2) / 2)
d = (ae['smape'].mean() - aelg['smape'].mean()) / pooled_std

print("AE vs AELG (Haar, ld=16)")
print(f"  MWU: U={u_stat}, p={p_mwu:.4f}")
print(f"  Wilcoxon: W={w_stat}, p={p_w:.4f}")
print(f"  Cohen's d = {d:.3f}")
print(f"  AE: {ae['smape'].mean():.3f} vs AELG: {aelg['smape'].mean():.3f}")
print(f"  --> No significant difference. AE has a negligible 0.06 SMAPE advantage.")
print()

# Haar AE vs SOTA
u2, p2 = stats.mannwhitneyu(ae['smape'].values, sota['smape'].values, alternative='two-sided')
d2 = (ae['smape'].mean() - sota['smape'].mean()) / np.sqrt((ae['smape'].std(ddof=1)**2 + sota['smape'].std(ddof=1)**2) / 2)
print("AE_haar vs SOTA (coif3 AELG)")
print(f"  MWU: U={u2}, p={p2:.4f}")
print(f"  Cohen's d = {d2:.3f} (medium-large)")
print(f"  AE_haar: {ae['smape'].mean():.3f} vs SOTA: {sota['smape'].mean():.3f}, delta=+{ae['smape'].mean()-sota['smape'].mean():.3f}")
print(f"  --> p=0.104, not significant at alpha=0.05 but strong trend. Effect size is medium-large.")
print()

# Haar AELG vs SOTA
u3, p3 = stats.mannwhitneyu(aelg['smape'].values, sota['smape'].values, alternative='two-sided')
d3 = (aelg['smape'].mean() - sota['smape'].mean()) / np.sqrt((aelg['smape'].std(ddof=1)**2 + sota['smape'].std(ddof=1)**2) / 2)
print("AELG_haar vs SOTA (coif3 AELG)")
print(f"  MWU: U={u3}, p={p3:.4f}")
print(f"  Cohen's d = {d3:.3f} (large)")
print(f"  AELG_haar: {aelg['smape'].mean():.3f} vs SOTA: {sota['smape'].mean():.3f}, delta=+{aelg['smape'].mean()-sota['smape'].mean():.3f}")
print(f"  --> Significantly worse than SOTA at alpha=0.05.")
print()

# Bootstrap probability
np.random.seed(42)
n_boot = 100000
ae_boots = np.random.choice(ae['smape'].values, size=(n_boot, 10), replace=True).mean(axis=1)
aelg_boots = np.random.choice(aelg['smape'].values, size=(n_boot, 10), replace=True).mean(axis=1)
sota_boots = np.random.choice(sota['smape'].values, size=(n_boot, 10), replace=True).mean(axis=1)
print("Bootstrap (100k resamples):")
print(f"  P(AE_haar < SOTA) = {(ae_boots < sota_boots).mean():.4f}")
print(f"  P(AELG_haar < SOTA) = {(aelg_boots < sota_boots).mean():.4f}")
print(f"  --> Only 5.8% and 1.2% chance respectively that Haar configs are truly better.")

## Latent Dim Scaling: ld=8 vs ld=12 vs ld=16 for Haar AE

Does ld=16 continue the improvement trend from the v2 study (where ld=12 > ld=8)?

In [ ]:
haar_v2 = v2_df[(v2_df['wavelet_family'] == 'haar') & (v2_df['block_type'] == 'TrendWaveletAE')]

ld_data = {}
for ld in sorted(haar_v2['latent_dim_cfg'].unique()):
    ld_data[f'ld={ld}'] = haar_v2[haar_v2['latent_dim_cfg'] == ld]['smape'].values
ld_data['ld=16'] = ae['smape'].values

fig, ax = plt.subplots(figsize=(8, 5))
positions = range(len(ld_data))
bp = ax.boxplot([ld_data[k] for k in ld_data.keys()], positions=positions, widths=0.6, patch_artist=True)
colors_ld = ['#3498db', '#e74c3c', '#2ecc71']
for patch, color in zip(bp['boxes'], colors_ld):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_xticks(positions)
ax.set_xticklabels(ld_data.keys())
ax.set_ylabel('SMAPE')
ax.set_title('Haar TrendWaveletAE: Latent Dim Scaling on Tourism-Yearly')
ax.axhline(y=sota['smape'].mean(), color='gray', linestyle='--', alpha=0.5, label=f'SOTA mean ({sota["smape"].mean():.3f})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('experiments/analysis/notebooks/tourism_haar_ld_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

for name, vals in ld_data.items():
    lo, hi = ci95(vals) if len(vals) > 1 else (vals.mean(), vals.mean())
    print(f"  {name}: N={len(vals)}, Mean={vals.mean():.3f}, Std={vals.std(ddof=1):.3f}, 95%CI=[{lo:.3f}, {hi:.3f}]")

# ld=16 vs ld=8
u, p = stats.mannwhitneyu(ld_data['ld=16'], ld_data['ld=8'], alternative='two-sided')
print(f"\n  ld=16 vs ld=8: MWU U={u}, p={p:.4f}")
# ld=16 vs ld=12
u, p = stats.mannwhitneyu(ld_data['ld=16'], ld_data['ld=12'], alternative='two-sided')
print(f"  ld=16 vs ld=12: MWU U={u}, p={p:.4f}")
print("\n  ld=16 is clearly the best latent dim for Haar on Tourism. The improvement is monotonic")
print("  and variance drops dramatically (std 0.17 at ld=16 vs 0.89 at ld=8, 1.75 at ld=12).")

## Conclusions

**SOTA unchanged.** TrendWaveletAELG_coif3_eq_bcast_td3_ld16 remains the Tourism-Yearly champion at SMAPE=20.864 (95% CI [20.712, 21.016]).

**Key findings:**
1. Haar at ld=16 does NOT beat coif3 at ld=16. The AE variant (SMAPE=20.996) is the closer of the two but still +0.132 above SOTA (MWU p=0.104, bootstrap P(better)=5.8%).
2. AE and AELG backbones are equivalent on Tourism at ld=16 (Wilcoxon p=0.32), confirming prior v2 study finding.
3. ld=16 is definitively the best latent dim for Haar on Tourism (MWU p=0.0002 vs ld=12), but the improvement saturates before reaching coif3 quality.

**Next experiments (priority order):**
1. Non-AE Trend + HaarWaveletV3 (no bottleneck) -- if M4 pattern holds, could leap to SOTA
2. TrendWaveletAE_coif3_eq_bcast_td3_ld16 -- swap AELG to AE on the winning config
3. Alternating TrendAE + HaarWaveletV3AE at ld=16 -- test depth-sensitivity on Tourism
4. Haar_eq_bcast vs Haar_eq_fcast at ld=16 -- isolate whether coif3's advantage is the wavelet or the larger basis_dim